# players.json 검증 (PLAN.md Phase 1)

**출력이 있는 상태로 커밋하지 말 것** — 실제 선수 데이터가 셀 출력에 들어간다 (핵심 규칙 1).
실행은 출력 파일을 gitignored 인 `data/` 에 쓰도록 한다:

```
.venv/Scripts/jupyter nbconvert --to notebook --execute pipeline/notebooks/validate.ipynb --output-dir data --output validate.2026.ipynb
```

In [ ]:
SEASON = 2026

import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pipeline").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "pipeline"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from model import DIST_90FT_M, time_to_distance
from params import build

from matplotlib import font_manager

_fonts = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"] if f in _fonts), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False

ss, meta = build(SEASON)
ss["t100"] = [time_to_distance(100, v, t) for v, t in zip(ss["vmax"], ss["tau"])]
ss["t30"] = [time_to_distance(30, v, t) for v, t in zip(ss["vmax"], ss["tau"])]
print(f"START_OFFSET={meta['startOffset']:+.4f}s  좌타 계수={meta['leftyCoef']:+.4f}s  스위치히터={meta['switchHitterSide']}")

## 1. 추정 100m 기록 분포 (반응 시간 제외)
목표: 최상위 11초대 후반~12초 초반, 평균 13초 안팎

In [ ]:
print(ss["t100"].describe().round(3).to_string())
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(ss["t100"], bins=40, color="#3b6ea5")
ax.axvline(ss["t100"].mean(), color="k", ls="--", lw=1, label=f"평균 {ss['t100'].mean():.2f}s")
ax.set_xlabel("추정 100m (s, 반응 시간 제외)"); ax.set_ylabel("선수 수"); ax.legend()
plt.show()

## 2. 상위 20명

In [ ]:
top = ss.sort_values("t100").head(20)
top[["fullName", "team", "nation", "batSide", "speed", "tau", "tauImputed", "sigmaScale", "t30", "t100"]].round(3).reset_index(drop=True)

## 3. 타석 방향별 τ 분포 (보정 전후)
목표: 보정 후 좌우타 τ 중앙값 차이 0.03초 이내. 결측 추정 선수는 제외.

In [ ]:
m = ss[~ss["tauImputed"]]
summary = m.groupby("batSide").agg(n=("tau", "size"), tau_raw_median=("tauRaw", "median"), tau_median=("tau", "median")).round(4)
print(summary.to_string())
diff_raw = abs(summary.loc["L", "tau_raw_median"] - summary.loc["R", "tau_raw_median"])
diff_adj = abs(summary.loc["L", "tau_median"] - summary.loc["R", "tau_median"])
print(f"L-R 중앙값 차이: 보정 전 {diff_raw:.4f}s → 보정 후 {diff_adj:.4f}s")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
for ax, col, title in [(axes[0], "tauRaw", "보정 전"), (axes[1], "tau", "보정 후")]:
    ax.boxplot([m.loc[m["batSide"] == s, col] for s in "LRS"], tick_labels=list("LRS"))
    ax.set_title(title); ax.set_ylabel("τ (s)")
plt.show()

## 4. 스프린트 스피드 대 추정 100m

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for imputed, g in ss.groupby("tauImputed"):
    ax.scatter(g["speed"], g["t100"], s=10, alpha=0.7, label="τ 추정값" if imputed else "τ 피팅")
ax.set_xlabel("스프린트 스피드 (ft/s)"); ax.set_ylabel("추정 100m (s)"); ax.legend(); ax.invert_yaxis()
plt.show()

## 5. τ 결측 비율과 피팅 경계값

In [ ]:
print(f"τ 추정(스플릿 없음): {ss['tauImputed'].sum()}명 / {len(ss)}명 = {ss['tauImputed'].mean():.1%}")
print(f"τ 경계값으로 잘린 선수: {ss['tauFlag'].notna().sum()}명")
print(f"τ 중앙값 {ss.loc[~ss['tauImputed'], 'tau'].median():.4f} (목표 1.1), 표준편차 {ss['tau'].std():.4f}")

## 6. sigmaScale (개인 기복) 분포

In [ ]:
print(ss["sigmaScale"].describe().round(3).to_string())
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(ss["sigmaScale"], bins=40, color="#a55b3b"); ax.set_xlabel("sigmaScale")
plt.show()

## 7. sample.py 분포 갱신용 요약 (평균·표준편차만 사용)

In [ ]:
print(f"sprint speed: mean={ss['speed'].mean():.2f} sd={ss['speed'].std():.2f} min={ss['speed'].min()} max={ss['speed'].max()}")
print(f"tau (피팅값): mean={m['tau'].mean():.3f} sd={m['tau'].std():.3f}")
print(f"batSide 비율: {ss['batSide'].value_counts(normalize=True).round(3).to_dict()}")
print(f"split 결측 비율: {ss['tauImputed'].mean():.3f}")